# 04 — Analytics views for Genie / AI-BI (Ayurveda chatbot)

**What this notebook does**  
Creates compact views so **Genie**, dashboards, or agents query stable names. **Primary** context: **`ayurgenix_curated`** (AyurGenix CSV). **Secondary** context: **`pdf_text_chunks`** (extracted Ayurveda PDF text from notebook 03).

| View | Purpose |
|------|--------|
| `vw_ayurgenix_sample` | Sample rows from curated AyurGenix (primary) |
| `vw_rag_pdf_catalog` | PDF files and Volume paths |
| `vw_rag_pdf_chunks_sample` | Sample of extracted PDF text chunks (secondary) |
| `vw_rag_pdf_chunks_by_file` | Chunk counts and character totals per PDF |
| `vw_pdf_extraction_quality` | **Per-PDF usefulness:** substantive chunk counts, total chars, % substantive |
| `vw_chatbot_structured_stats` | Total curated CSV rows |
| `vw_chatbot_knowledge_footprint` | One-row metrics: CSV rows, PDF files, PDF chunks, total PDF chars |

**How to run** — Serverless → **Run all** (requires **`pdf_text_chunks`** from notebook 03). Use **`vw_pdf_extraction_quality`** to confirm PDFs yielded useful text before spending on embeddings.


In [0]:
%sql
USE ayurveda_lakehouse.ayurgenix;


In [0]:
%sql
CREATE OR REPLACE VIEW vw_ayurgenix_sample AS
SELECT * FROM ayurveda_lakehouse.ayurgenix.ayurgenix_curated LIMIT 500;


In [0]:
%sql
CREATE OR REPLACE VIEW vw_rag_pdf_catalog AS
SELECT file_name, volume_path FROM ayurveda_lakehouse.ayurgenix.rag_pdf_manifest;


In [0]:
%sql
CREATE OR REPLACE VIEW vw_rag_pdf_chunks_sample AS
SELECT source_file, page_number, chunk_index, char_count, chunk_text
FROM ayurveda_lakehouse.ayurgenix.pdf_text_chunks
WHERE length(trim(chunk_text)) > 0
LIMIT 200;


In [0]:
%sql
CREATE OR REPLACE VIEW vw_rag_pdf_chunks_by_file AS
SELECT source_file,
       COUNT(*) AS chunk_rows,
       SUM(char_count) AS total_chars,
       COUNT(DISTINCT page_number) AS pages_seen
FROM ayurveda_lakehouse.ayurgenix.pdf_text_chunks
GROUP BY source_file
ORDER BY source_file;


In [0]:
%sql
CREATE OR REPLACE VIEW vw_pdf_extraction_quality AS
SELECT
  source_file,
  COUNT(*) AS total_chunks,
  SUM(CASE WHEN length(trim(chunk_text)) >= 50 THEN 1 ELSE 0 END) AS chunks_with_useful_text,
  SUM(char_count) AS total_chars,
  ROUND(100.0 * SUM(CASE WHEN length(trim(chunk_text)) >= 50 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_substantive
FROM ayurveda_lakehouse.ayurgenix.pdf_text_chunks
GROUP BY source_file
ORDER BY source_file;


In [0]:
%sql
CREATE OR REPLACE VIEW vw_chatbot_structured_stats AS
SELECT COUNT(*) AS ayurgenix_curated_rows FROM ayurveda_lakehouse.ayurgenix.ayurgenix_curated;


In [0]:
%sql
CREATE OR REPLACE VIEW vw_chatbot_knowledge_footprint AS
SELECT 'curated_csv_rows' AS metric, CAST(COUNT(*) AS STRING) AS value
FROM ayurveda_lakehouse.ayurgenix.ayurgenix_curated
UNION ALL
SELECT 'pdf_manifest_files', CAST(COUNT(*) AS STRING) FROM ayurveda_lakehouse.ayurgenix.rag_pdf_manifest
UNION ALL
SELECT 'pdf_text_chunks', CAST(COUNT(*) AS STRING) FROM ayurveda_lakehouse.ayurgenix.pdf_text_chunks
UNION ALL
SELECT 'pdf_text_chars', CAST(SUM(char_count) AS STRING) FROM ayurveda_lakehouse.ayurgenix.pdf_text_chunks;


## Sanity check — each view returns rows


In [0]:
%sql
SELECT 'vw_ayurgenix_sample' AS view_name, COUNT(*) AS row_count FROM vw_ayurgenix_sample
UNION ALL SELECT 'vw_rag_pdf_catalog', COUNT(*) FROM vw_rag_pdf_catalog
UNION ALL SELECT 'vw_rag_pdf_chunks_sample', COUNT(*) FROM vw_rag_pdf_chunks_sample
UNION ALL SELECT 'vw_rag_pdf_chunks_by_file', COUNT(*) FROM vw_rag_pdf_chunks_by_file
UNION ALL SELECT 'vw_pdf_extraction_quality', COUNT(*) FROM vw_pdf_extraction_quality
UNION ALL SELECT 'vw_chatbot_structured_stats', COUNT(*) FROM vw_chatbot_structured_stats
UNION ALL SELECT 'vw_chatbot_knowledge_footprint', COUNT(*) FROM vw_chatbot_knowledge_footprint;


**Next step:** wire retrieval: **answer from `ayurgenix_curated` first**, then enrich with **`pdf_text_chunks`**. Re-check **`vw_pdf_extraction_quality`** after swapping PDFs.
